In [1]:
# %%

#------------------------------------------------ Begin_Librairie ----------------------------------------

import datetime

import pandas as pd

from pandas import ExcelWriter

from selenium import webdriver

from time import sleep

import os

from selenium.webdriver.common.by import By

from bs4 import BeautifulSoup



import os

import re



# %%

#------------------------------------------------ Begin_ fileName ----------------------------------------

regulatorName = 'RO NBRO' ## change to current controller name



print(f"Running {regulatorName} Web Scraping Tool v.1.1")

now=datetime.datetime.now()

filename = '{} data {}.xlsx'.format(regulatorName, str(now).replace(":",".")[:-7])

#scriptfolder = f"C:\\Users\\siewekoa\\OneDrive - moodys.com\\Desktop\\My_data\\Project_work\\scripts_regulator\\{regulatorName}" ## to comment for the production environment
scriptfolder = f"C:\\Users\\wuj1\\OneDrive - Moody's\\Desktop\\Regulator\\{regulatorName}"

#scriptfolder=os.path.dirname(os.path.abspath(__file__)) ## to decomment for the production environment

os.chdir(scriptfolder)

writer = ExcelWriter(filename)

tempfolder=os.path.join(scriptfolder, 'tempfolder') #if files are downloaded during the process



if os.path.exists(tempfolder):

    for rem in os.listdir(tempfolder):

        os.remove(os.path.join(tempfolder, rem))

else:

    os.mkdir(tempfolder)



# %%

#------------------------------------------------ Begin_chromedriver ----------------------------------------

#Starting Chrome driver, set to download files in tempfolder

chromeOptions = webdriver.ChromeOptions()

prefs = {"plugins.always_open_pdf_externally": True,

		 "download.prompt_for_download": False,

		 "download.default_directory" : tempfolder,

         'profile.default_content_setting_values.automatic_downloads': 1 # Desable a Multiplefile download alert

         }

chromeOptions.add_argument("--disable-search-engine-choice-screen")

chromeOptions.add_experimental_option("prefs",prefs)

driver = webdriver.Chrome(options=chromeOptions)

driver.maximize_window()



# %%

#------------------------------------------------ Begin_Variable ----------------------------------------

regdict = { 'RO NBRO 1': 'https://www.bnro.ro/I.-REGISTER-OF-CREDIT-INSTITUTIONS-25352.aspx#IC_Banci_Active', 

            'RO NBRO 2': 'https://www.bnro.ro/I.-REGISTER-OF-CREDIT-INSTITUTIONS-25352.aspx#IC_BECDL_Active', 

            'RO NBRO 3': 'https://www.bnro.ro/I.-REGISTER-OF-CREDIT-INSTITUTIONS-25352.aspx#IC_OCC1_Active', 

            'RO NBRO 4': 'https://www.bnro.ro/I.-REGISTER-OF-CREDIT-INSTITUTIONS-25353.aspx#IC_SASM_Active',

            'RO NBRO 5': 'https://www.bnro.ro/III.-REGISTER-OF-PAYMENT-INSTITUTIONS-25132.aspx',

            'RO NBRO 6': 'https://www.bnro.ro/IV.-REGISTER-OF-ELECTRONIC-MONEY-INSTITUTIONS--25296.aspx',

            'RO NBRO 7': 'https://www.bnro.ro/V.-REGISTRUL-INSTITU%c8%9aIILOR-FINANCIARE-NEBANCARE-(IFN)-25274.aspx',

            'RO NBRO 8': 'https://www.bnro.ro/V.-REGISTRUL-INSTITU%c8%9aIILOR-FINANCIARE-NEBANCARE-(IFN)-25292.aspx',

            'RO NBRO 9': 'https://www.bnro.ro/files/d/RegistreBNR/ifn/RegistrulDeEvidenta/registrul_evidenta_ifn_active_tot.htm',

            'RO NBRO 11': 'https://www.bnro.ro/DocumentInformation.aspx?idDocument=11312&directLink=1',

            'RO NBRO 12': 'https://www.bnro.ro/DocumentInformation.aspx?idDocument=11314&directLink=1',

            'RO NBRO 13': 'https://www.bnro.ro/DocumentInformation.aspx?idDocument=11318&directLink=1'

            }



Id_reglist = {

    'RO NBRO 1': ['IC_Banci_Active'],

    'RO NBRO 2': ['IC_BECDL_Active'],

    'RO NBRO 3': ['IC_OCC1_Active', 'IC_OCC2_Active'],

    'RO NBRO 4': ['IC_SASM_Active'],

    'RO NBRO 5': ['IP_Active'],

    'RO NBRO 6': ['IEME_Active'],

    'RO NBRO 7': ['IFN_A_S1', 'IFN_A_S2', 'IFN_A_S3', 'IFN_A_S4', 'IFN_A_S5', 'IFN_A_S6', 'IFN_A_S7', 'IFN_A_S12', 'IFN_A_S8', 'IFN_A_S9', 'IFN_A_S10', 'IFN_A_M'],

    'RO NBRO 8': ['IFN_RS_A_S1', 'IFN_RS_A_S2', 'IFN_RS_A_S3', 'IFN_RS_A_S4', 'IFN_RS_A_S5', 'IFN_RS_A_S6', 'IFN_RS_A_S7', 'IFN_RS_A_S12', 'IFN_RS_A_S8', 'IFN_RS_A_S9', 'IFN_RS_A_S10', 'IFN_RS_A_M'],

    'RO NBRO 9': ['IP_Active'],

    'RO NBRO 11': [4, -6],

    'RO NBRO 12': [3, -3],

    'RO NBRO 13': [1, -2],

    }



sqldict={'bvdid': [], 'priority': [], 'ListLabel': [], 'Typology': [], 'EntryType': [], 'Name': [], 'InternalID_1': [], 'InternalID_1_type': [], 'InternalID_2': [], 

          'InternalID_2_type': [], 'InternalID_3': [], 'InternalID_3_type': [], 'CoType': [], 'License_Type': [], 'Address_1': [], 'Address_2': [], 'City': [], 

          'Zip': [], 'Cntry': [], 'Phone': [], 'Fax': [], 'Website': [], 'Email': [], 'RegulationType': [], 'RegulationTypeCode': [], 'RegulationDate': [], 'CancellationDate': [], 

          'RegCtry': [], 'RegCode' : [], 'ListCode': [], 'ListLanguage': [], 'ListValidityDate': [], 'ListName': [], 'ListProcessDate': [], 'LEI Code': [], 'BIC SWIFT Code': [], 'Name - Mother Company': [],

          'Address_1 - Mother company': [], 'Address_2 -  Mother company': [], 'City - Mother company': [], 'Zip - Mother company': [], 'Cntry - Mother company': [], 

          'Phone - Mother company': [], 'Check': []}



processdate = now.strftime('%Y-%m-%d')



# %%

#------------------------------------------------ Begin_Fouction ----------------------------------------

def bourange_same_length_array(sqldict) :

    len_value=[]

    for key, value in sqldict.items():

        len_value.append(len(value))

    maxlen = max(len_value)

    for key, val in sqldict.items():

        if len(sqldict[key]) != maxlen:

            empty = []

            total_empty = maxlen - len(sqldict[key])

            for i in range(total_empty):

                empty.append('')

            sqldict[key]=sqldict[key]+empty

    return sqldict



def check_dowload_files(tempfolder, fileType, wait_time=10):

    for time in range(wait_time):

        if len([ele for ele in os.listdir(tempfolder) if '.crdownload' not in ele and '.tmp' not in ele]) != 0 :

            print(f"[INFO] : {fileType} file = {os.listdir(tempfolder)})")

            break

        else:

            print(f"[INFO] : Download {fileType} file ... (wait {time*2}/20 s)")

            sleep(2)

    else:

        raise Exception(f'[ERROR] : Failed to Download {fileType} file. Run Script again' )

    return  os.listdir(tempfolder)[0]

    

def search_expressions(text):

    pattern = r'\[#\d+#\]'

    matches = re.findall(pattern, text)

    return matches



# %%

for k, reg in enumerate(regdict):



    print(f"[INFO] : Regulator list {k + 1}/{len(regdict)} | {reg} ")

    driver.get(regdict[reg])

    sleep(6)

    soup = BeautifulSoup(driver.page_source, "html.parser")



    if reg.split()[-1] in ['1', '2', '3', '4', '5', '6', '7', '8']:

        contentDiv = soup.find("div",{"id":"contentDiv"})

        

        for iD in Id_reglist[reg] :

            IC_regName = contentDiv.find("div",{"id":f"{iD}"})

            try:Typology = IC_regName.find_all("h3")[-1].text.split('–')[-1].strip()

            except:Typology = ''



            if len(search_expressions(IC_regName.text)) == 0:

                all_tr = IC_regName.find("tbody").find_all("tr")

                print(f"[INFO] : -- Reg {k+1}/{len(regdict)} | Company = {len(all_tr)} ")



                for i, tr in enumerate(all_tr):

                    all_td = tr.find_all("td")

                    

                    if reg == 'RO NBRO 1' or reg == 'RO NBRO 2' or reg == 'RO NBRO 3' or reg == 'RO NBRO 7':

                        sqldict['Name'].append(all_td[2].text)

                        sqldict['InternalID_1'].append(all_td[0].text)

                        sqldict['InternalID_1_type'].append("Number")

                        sqldict['Typology'].append(Typology)

                        sqldict['Address_1'].append(all_td[3].text)

                        sqldict['InternalID_2'].append(all_td[4].text)

                        sqldict['InternalID_2_type'].append("Tax identification number")

                        sqldict['InternalID_3'].append(all_td[5].text)

                        sqldict['InternalID_3_type'].append("Trade register office number")

                        sqldict['LEI Code'].append(all_td[6].text)

                        # sqldict['RegulationDate'].append(all_td[7].text)

                    elif reg == 'RO NBRO 4' :

                        sqldict['Name'].append(all_td[2].text)

                        sqldict['InternalID_1'].append(all_td[0].text)

                        sqldict['InternalID_1_type'].append("Number")

                        sqldict['Address_1'].append(all_td[3].text)

                        sqldict['LEI Code'].append(all_td[4].text)

                        # sqldict['RegulationDate'].append(all_td[5].text)

                        sqldict['Typology'].append(Typology)

                    elif reg == 'RO NBRO 5' or reg == 'RO NBRO 6':

                        sqldict['Name'].append(all_td[2].text)

                        sqldict['InternalID_1'].append(all_td[0].text)

                        sqldict['InternalID_1_type'].append("Number")

                        sqldict['Typology'].append(all_td[3].text)

                        sqldict['Address_1'].append(all_td[4].text)

                        sqldict['InternalID_2'].append(all_td[5].text)

                        sqldict['InternalID_2_type'].append("Tax identification number")

                        sqldict['InternalID_3'].append(all_td[6].text)

                        sqldict['InternalID_3_type'].append("Trade register office number")

                        sqldict['LEI Code'].append(all_td[7].text)

                    elif reg == 'RO NBRO 8' :

                        sqldict['Name'].append(all_td[4].text)

                        sqldict['Typology'].append(Typology)

                        sqldict['Address_1'].append(all_td[5].text)

                        sqldict['InternalID_1'].append(all_td[0].text)

                        sqldict['InternalID_1_type'].append("Special register Number")

                        sqldict['InternalID_2'].append(all_td[6].text)

                        sqldict['InternalID_2_type'].append("Tax identification number")

                        sqldict['InternalID_3'].append(all_td[7].text)

                        sqldict['InternalID_3_type'].append("Trade register office number")

                        sqldict['LEI Code'].append(all_td[8].text)

  

                    sqldict['RegulationType'].append('Licenced')

                    sqldict['ListProcessDate'].append(processdate)

                    sqldict['RegCtry'].append(reg.split(' ')[0])

                    sqldict['RegCode'].append(reg.split(' ')[1])

                    sqldict['ListCode'].append(reg.split(' ')[-1])

                        

                sqldict = bourange_same_length_array(sqldict)



    elif reg.split()[-1] in ['9'] :



        body = soup.find("body",{"class":"bs"})

        tables = body.find_all("table",{"border":"1"})

        for table in tables:

            trs = table.find_all("tr",{"valign":"top"})

            print(f"[INFO] : -- Reg {k+1}/{len(regdict)} | Company = {len(trs)} ")

            for tr in trs:

                sqldict['Name'].append(tr.find_all("td")[2].text.strip())

                sqldict['Address_1'].append(tr.find_all("td")[3].text.strip())              

                sqldict['InternalID_1'].append(tr.find_all("td")[0].text.strip())

                sqldict['InternalID_1_type'].append('Înscrierea în Registrul de Evidenţă')

                sqldict['InternalID_2'].append(tr.find_all("td")[4].text.strip())

                sqldict['InternalID_2_type'].append('Cod unic de înregistrare')

                sqldict['InternalID_3'].append(tr.find_all("td")[5].text.strip())

                sqldict['InternalID_3_type'].append('Înmatriculare în Registrul Comerţului')

                

                sqldict['RegulationType'].append('Supervised')

                sqldict['ListProcessDate'].append(processdate)

                sqldict['RegCtry'].append(reg.split(' ')[0])

                sqldict['RegCode'].append(reg.split(' ')[1])

                sqldict['ListCode'].append(reg.split(' ')[-1])

        

            sqldict = bourange_same_length_array(sqldict)



    elif reg.split()[-1] in ['11', '12', '13'] :

        file =  check_dowload_files(tempfolder, "csv" )

        filePath = os.path.join(tempfolder, file)

        columns = ['Num', 'Names'] # A definir...

        df = pd.read_excel(filePath)

        df.columns = columns

        df = df.fillna("")  # subsitute nan with empty strings

        df = df[Id_reglist[reg][0]:Id_reglist[reg][1]]

        df = df.reset_index(drop=True)

        print(f"[INFO] : -- DataFrame '{file}' | containe = {df.shape}")

        for index, row in df.iterrows():

            sqldict['Name'].append(row['Names'])            

            sqldict['RegulationType'].append('Supervised')

            sqldict['ListProcessDate'].append(processdate)

            sqldict['RegCtry'].append(reg.split(' ')[0])

            sqldict['RegCode'].append(reg.split(' ')[1])

            sqldict['ListCode'].append(reg.split(' ')[-1])

        sqldict = bourange_same_length_array(sqldict)



# %%

#------------------------------------------------ Begin_writer and save df to excel  ----------------------------------------

os.chdir(scriptfolder)

df=pd.DataFrame(sqldict)

df.to_excel(writer, 'SQL Ready', index=False)

writer.save()

writer.close()

driver.quit()

sleep(3)
    

Running RO NBRO Web Scraping Tool v.1.1
[INFO] : Regulator list 1/12 | RO NBRO 1 


AttributeError: 'NoneType' object has no attribute 'find'